In [0]:
from pyspark.sql import functions as F
price_df = (
    spark.read
    .format("csv")
    .option("inferSchema", True)
    .option("header", True)
    .load("/Volumes/business_to_business/sports_bar_data/gross_price/*")
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
display(price_df)

product_id,month,gross_price,ingestion_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-09-06T05:04:19.937Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-09-06T05:04:19.937Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-09-06T05:04:19.937Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-09-06T05:04:19.937Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-09-06T05:04:19.937Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-09-06T05:04:19.937Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-09-06T05:04:19.937Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-09-06T05:04:19.937Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-09-06T05:04:19.937Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-09-06T05:04:19.937Z,gross_price.csv,2741


In [0]:
price_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
price_df.write\
.format("delta")\
.option("delta.enableChangeDataFeed", "true")\
.option("mergeSchema", "true")\
.mode("overwrite")\
.saveAsTable("business_to_business.bronze.pricing_data")

###  Cleansing Data 

In [0]:
silver_df = spark.read.table("business_to_business.bronze.pricing_data")

In [0]:
silver_df.show()

+----------+----------+-------------+--------------------+---------------+---------+
|product_id|     month|  gross_price| ingestion_timestamp|      file_name|file_size|
+----------+----------+-------------+--------------------+---------------+---------+
|  25891101|2025/07/01|          -84|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891101|01/08/2025|      unknown|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891101|2025/09/01|           84|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891101|2025-10-01|           83|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891101|2025-11-01|           83|2026-09-06 05:04:...|gross_price.csv|     2741|
|  88888888|2025-12-01|          -83|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891102|2025-07-01|           68|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891102|2025-08-01|           68|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891102|2025-09-01|           68|2026-09-06 05:04:...|gross_p

In [0]:
silver_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
silver_df.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025/07/01|
|01/08/2025|
|2025/09/01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025/11/01|
|2025/08/01|
|01-09-2025|
|2025/10/01|
|01/12/2025|
|01/09/2025|
|01-12-2025|
|01-08-2025|
|01/10/2025|
+----------+



In [0]:
# 1️. Parse `month` from multiple possible formats
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]


In [0]:
silver_df = silver_df.withColumn("month", (
    F.coalesce(
     F.try_to_date(F.col("month"), "yyyy/MM/dd")
    ,F.try_to_date(F.col("month"), "dd/MM/yyyy")
    ,F.try_to_date(F.col("month"), "yyyy-MM-dd")
    ,F.try_to_date(F.col("month"), "dd-MM-yyyy")

    )
))

In [0]:
silver_df.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
+----------+



In [0]:
display(silver_df)


product_id,month,gross_price,ingestion_timestamp,file_name,file_size
25891101,2025-07-01,-84,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,2025-08-01,unknown,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,2025-09-01,84,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-09-06T05:04:27.933Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-09-06T05:04:27.933Z,gross_price.csv,2741


In [0]:
# Adjusting the gross price values
silver_df = (
    silver_df.withColumn("gross_price", 
                            F.when(F.col("gross_price").rlike(r"^-?\d+(\.\d+)?$"),
                                   F.when(F.col("gross_price").cast("double") <0, -1*F.col("gross_price").cast("double"))
                                   .otherwise(F.col("gross_price").cast("double"))
                                   )
                            .otherwise(0)
                         )
)

In [0]:
display(silver_df)

product_id,month,gross_price,ingestion_timestamp,file_name,file_size
25891101,2025-07-01,84.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,2025-08-01,0.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,2025-09-01,84.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,2025-10-01,83.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,2025-11-01,83.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
88888888,2025-12-01,83.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,2025-07-01,68.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,2025-08-01,68.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,2025-09-01,68.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,2025-10-01,69.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741


In [0]:
silver_products = spark.read.table("business_to_business.silver.dim_sbproducts")

In [0]:
silver_df = silver_df.join(silver_products.select(["product_id","product_code"]), on="product_id", how="inner")

In [0]:
silver_df.show()

+----------+----------+-----------+--------------------+---------------+---------+--------------------+
|product_id|     month|gross_price| ingestion_timestamp|      file_name|file_size|        product_code|
+----------+----------+-----------+--------------------+---------------+---------+--------------------+
|  25891101|2025-07-01|       84.0|2026-09-06 05:04:...|gross_price.csv|     2741|521fcd441ab9d975c...|
|  25891101|2025-08-01|        0.0|2026-09-06 05:04:...|gross_price.csv|     2741|521fcd441ab9d975c...|
|  25891101|2025-09-01|       84.0|2026-09-06 05:04:...|gross_price.csv|     2741|521fcd441ab9d975c...|
|  25891101|2025-10-01|       83.0|2026-09-06 05:04:...|gross_price.csv|     2741|521fcd441ab9d975c...|
|  25891101|2025-11-01|       83.0|2026-09-06 05:04:...|gross_price.csv|     2741|521fcd441ab9d975c...|
|  25891102|2025-07-01|       68.0|2026-09-06 05:04:...|gross_price.csv|     2741|7f6658d62c9204ef7...|
|  25891102|2025-08-01|       68.0|2026-09-06 05:04:...|gross_pr

In [0]:
silver_df = silver_df.select(
    "product_id"
    ,"product_code"
    ,"month"
    ,"gross_price"
    ,"ingestion_timestamp"
    ,"file_name"
    ,"file_size"
)
silver_df.show()


+----------+--------------------+----------+-----------+--------------------+---------------+---------+
|product_id|        product_code|     month|gross_price| ingestion_timestamp|      file_name|file_size|
+----------+--------------------+----------+-----------+--------------------+---------------+---------+
|  25891101|521fcd441ab9d975c...|2025-07-01|       84.0|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891101|521fcd441ab9d975c...|2025-08-01|        0.0|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891101|521fcd441ab9d975c...|2025-09-01|       84.0|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891101|521fcd441ab9d975c...|2025-10-01|       83.0|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891101|521fcd441ab9d975c...|2025-11-01|       83.0|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891102|7f6658d62c9204ef7...|2025-07-01|       68.0|2026-09-06 05:04:...|gross_price.csv|     2741|
|  25891102|7f6658d62c9204ef7...|2025-08-01|       68.0|2026-09-

In [0]:
silver_df.write\
.mode("overwrite")\
.format("delta")\
.option("delta.enableChangeDataFeed", "true")\
.option("mergeSchema","true")\
.saveAsTable("business_to_business.silver.pricing_data")

### Gold Layer Operations

In [0]:
gold_df = spark.read.table('business_to_business.silver.pricing_data')
display(gold_df)

product_id,product_code,month,gross_price,ingestion_timestamp,file_name,file_size
25891101,521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,2025-07-01,84.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,2025-08-01,0.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,2025-09-01,84.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,2025-10-01,83.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891101,521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,2025-11-01,83.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,2025-07-01,68.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,2025-08-01,68.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,2025-09-01,68.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,2025-10-01,69.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741
25891102,7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,2025-11-01,69.0,2026-09-06T05:04:27.933Z,gross_price.csv,2741


In [0]:
# select the required Fields
gold_df = gold_df.select(
    "product_code"
    ,F.col("gross_price")
    ,F.col("month")
    
)


In [0]:
gold_df.write\
.mode("overwrite")\
.format("delta")\
.option("mergeSchema", "true")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable("business_to_business.gold.dim_sbgross_price")

### Merge the new records 

In [0]:
# from delta.tables import DeltaTable
# delta_table = DeltaTable.forName(spark, "business_to_business.gold.dim_sbgross_price")
df_gold_price = spark.read.table("business_to_business.gold.dim_sbgross_price")
display(df_gold_price)


product_code,gross_price,month
521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,84.0,2025-07-01
521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,0.0,2025-08-01
521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,84.0,2025-09-01
521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,83.0,2025-10-01
521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,83.0,2025-11-01
7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,68.0,2025-07-01
7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,68.0,2025-08-01
7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,68.0,2025-09-01
7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,69.0,2025-10-01
7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,69.0,2025-11-01


In [0]:
df_gold_price = df_gold_price.withColumn("year",F.year(F.col("month")))\
.withColumn("is_zero", 
            F.when(F.col("gross_price") == 0, 1)
            .otherwise(0)
            )

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("product_code", "year").orderBy(F.col("is_zero"),F.col("month").desc())
df_gold_updated  = df_gold_price.withColumn("rank", F.rank().over(window_spec)).filter(F.col("rank") == 1)
# ordering the fields
df_gold_updated = df_gold_updated.select(
    F.col("product_code")
    ,F.col("gross_price").alias("price_inr")
    ,F.col("year")
)

In [0]:
df_gold_updated.withColumn("year",F.col("year").cast("string"))

DataFrame[product_code: string, price_inr: double, year: string]

In [0]:
df_gold_updated.printSchema()
source = df_gold_updated
source.show(truncate=False)

root
 |-- product_code: string (nullable = true)
 |-- price_inr: double (nullable = true)
 |-- year: integer (nullable = true)

+----------------------------------------------------------------+---------+----+
|product_code                                                    |price_inr|year|
+----------------------------------------------------------------+---------+----+
|11829ae9a1fba9dea2a44489773504db86d29211f09fa0e6f63bef020183cd0f|432.0    |2025|
|38b61b697918c0ad776064b712efaf00297aaa6b5b2b453d8347226c5050f46e|440.0    |2025|
|399205b103ee9f68f358ee32b7fa9d5ec6ca2784eba72ab051191d8bd87d7a95|300.0    |2025|
|521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913|83.0     |2025|
|52d9158987029d4fa6688257dd034d2fee33778414ada3babe86f74e0a63dfc3|493.0    |2025|
|53361f1d15f3967db2b9910b15dcb1e232f4bef98895594f616b4fde55548d39|50.0     |2025|
|6c7000a2708d3a2ec0b87e57daed46779b14f86207cb9737b6c28ae1813aaed8|86.0     |2025|
|7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "business_to_business.gold.dim_gross_price")
# source = df_gold_updated
delta_table.alias("t").merge(
    source = df_gold_updated.alias("s"),
    condition= "t.product_code = s.product_code"
).whenMatchedUpdate(
    set = {
        "price_inr": "s.price_inr",
        "year": "s.year"
    }
).whenNotMatchedInsert(
        values = {"product_code": "s.product_code",
                    "price_inr": "s.price_inr",
                    "year": "s.year"
                }
).execute()
# source records are not inserted

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]